## Step 1 - Import Libraries
Import the packages used for upload, document extraction, embeddings, Pinecone, and Ollama.
The environment variables are loaded from `.env`, and noisy warnings are hidden.

In [5]:
import os
import json
import time
import shutil
import hashlib
import warnings
import logging
from pathlib import Path

import gradio as gr
import numpy as np
import pandas as pd
import torch
import ollama
import pdfplumber
import pymupdf
import pytesseract

from PIL import Image
from docx import Document
from pptx import Presentation
from tqdm import tqdm
from dotenv import load_dotenv
from IPython import display
from pinecone import Pinecone, ServerlessSpec
from transformers import CLIPModel, CLIPProcessor
from sentence_transformers import CrossEncoder
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.ERROR)

print("Libraries loaded")

/home/radhey/Desktop/MAIN/web dev/multimodal-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded


## Step 2 - Create Folders
Create the folders where uploaded files and extracted content will be stored.
The same folders are reused each time you run the notebook.

In [6]:
BASE = Path("data")
UPLOAD_DIR = BASE / "uploads"

for folder in [
    UPLOAD_DIR,
    BASE / "text",
    BASE / "images",
    BASE / "tables",
    BASE / "ocr",
]:
    folder.mkdir(parents=True, exist_ok=True)

SUPPORTED_EXTENSIONS = {".pdf", ".docx", ".pptx", ".xlsx", ".xls", ".csv"}
filepaths = []

print("Folders ready")

Folders ready


## Step 3 - Define Upload Helpers
These helper functions save uploaded files into `data/uploads`.
They also keep track of the uploaded file paths for the rest of the pipeline.

In [7]:
def uploaded_path(file):
    if isinstance(file, str):
        return Path(file)
    return Path(file.name)


def uploaded_message(skipped=None):
    skipped = skipped or []

    if filepaths:
        message = "Uploaded documents:\n" + "\n".join(filepaths)
    else:
        message = "No supported documents uploaded yet."

    if skipped:
        message += "\n\nSkipped unsupported files:\n" + "\n".join(skipped)

    return message

## Step 4 - Show Upload UI
This cell displays the Gradio upload button.
Upload one or more supported documents before continuing to the next step.

In [8]:
def save_uploaded_documents(files):
    global filepaths

    files = [] if not files else files if isinstance(files, list) else [files]
    saved = []
    skipped = []

    for file in files:
        source = uploaded_path(file)

        if source.suffix.lower() not in SUPPORTED_EXTENSIONS:
            skipped.append(source.name)
            continue

        target = UPLOAD_DIR / source.name

        if source.resolve() != target.resolve():
            shutil.copy2(source, target)

        saved.append(str(target))

    filepaths = filepaths + [path for path in saved if path not in filepaths]
    return uploaded_message(skipped)


def clear_uploaded_documents():
    global filepaths
    filepaths = []
    return uploaded_message()


with gr.Blocks(title="Multimodal RAG Upload") as upload_ui:
    upload = gr.UploadButton(
        "Upload documents",
        file_count="multiple",
        file_types=sorted(SUPPORTED_EXTENSIONS),
    )
    clear = gr.Button("Clear uploaded list")
    output = gr.Textbox(label="Uploaded files", lines=10, value=uploaded_message())

    upload.upload(save_uploaded_documents, upload, output)
    clear.click(clear_uploaded_documents, None, output)

upload_ui.launch(inline=True, share=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Step 5 - Build Document Metadata
Convert uploaded file paths into structured document records.
Each document gets a stable hash ID for source tracking and citations.

In [9]:
if not filepaths:
    raise ValueError("Upload one or more supported documents before running this cell.")

documents = []

for path in filepaths:
    file_path = Path(path)
    document_id = hashlib.sha256(file_path.read_bytes()).hexdigest()

    documents.append({
        "filepath": str(file_path),
        "file_name": file_path.name,
        "extension": file_path.suffix.lower(),
        "document_id": document_id,
    })

print(f"Ready to process {len(documents)} document(s)")

for document in documents:
    print(document["file_name"], document["extension"], document["document_id"][:12])

Ready to process 2 document(s)
cc1.pdf .pdf ae0bfbc23b96
5 PPT Format.pptx .pptx a8908d15835d


## Step 6 - Preview First PDF
If the first uploaded document is a PDF, this cell previews it inline.
For Word, PowerPoint, Excel, and CSV files, the notebook continues without a preview.

In [10]:
first_document = documents[0]

if first_document["extension"] == ".pdf":
    display.display(
        display.IFrame(
            first_document["filepath"],
            width=1000,
            height=600,
        )
    )
else:
    print("Preview is only shown for PDFs")

## Step 7 - Define Chunk Helpers
These helpers split long text into smaller chunks and save each chunk to disk.
Every chunk keeps metadata such as file name, location, content type, and source path.

In [11]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=200,
    length_function=len,
)


def safe_name(value):
    return str(value).replace("/", "_").replace(" ", "_")


def add_chunks(items, text, document, location, item_type="text", folder="text"):
    if not str(text).strip():
        return

    for chunk_number, chunk in enumerate(text_splitter.split_text(str(text))):
        path = BASE / folder / f"{document['file_name']}_{item_type}_{safe_name(location)}_{chunk_number}.txt"
        path.write_text(chunk, encoding="utf-8")

        items.append({
            **document,
            "page": location,
            "location": location,
            "type": item_type,
            "text": chunk,
            "path": str(path),
        })

## Step 8 - Define Table Helpers
Tables from PDFs, Word, PowerPoint, Excel, and CSV files are converted into readable text.
This lets the same text embedding path handle both normal text and tables.

In [12]:
def table_to_text(rows):
    lines = []

    for row in rows:
        cells = ["" if cell is None else str(cell).strip().replace("\n", " ") for cell in row]
        lines.append(" | ".join(cells))

    return "\n".join(lines)


def dataframe_to_text(dataframe):
    dataframe = dataframe.dropna(how="all").dropna(axis=1, how="all")

    if dataframe.empty:
        return ""

    return dataframe.astype(str).to_csv(index=False, sep="|")

## Step 9 - Extract PDF Content
PDFs are processed for native text, OCR text, embedded images, and tables.
Images are saved separately so they can receive image embeddings later.

In [13]:
def extract_pdf(document, items):
    pdf = pymupdf.open(document["filepath"])

    with pdfplumber.open(document["filepath"]) as plumber:
        for page_number in tqdm(range(len(pdf)), desc=document["file_name"]):
            page = pdf[page_number]
            location = f"Page {page_number + 1}"

            add_chunks(items, page.get_text(), document, location)

            page_image = BASE / "ocr" / f"{document['file_name']}_page_{page_number:03d}.png"
            page.get_pixmap(dpi=200).save(page_image)
            ocr_text = pytesseract.image_to_string(Image.open(page_image))
            add_chunks(items, ocr_text, document, location, item_type="ocr", folder="ocr")

            for image_number, image in enumerate(page.get_images(full=True)):
                xref = image[0]
                pixmap = pymupdf.Pixmap(pdf, xref)

                if pixmap.n - pixmap.alpha > 3:
                    pixmap = pymupdf.Pixmap(pymupdf.csRGB, pixmap)

                image_path = BASE / "images" / f"{document['file_name']}_image_{page_number}_{image_number}_{xref}.png"
                pixmap.save(image_path)

                items.append({
                    **document,
                    "page": location,
                    "location": location,
                    "type": "image",
                    "path": str(image_path),
                })

            pdf_page = plumber.pages[page_number]

            for table_number, table in enumerate(pdf_page.extract_tables() or []):
                table_location = f"{location}, Table {table_number + 1}"
                add_chunks(items, table_to_text(table), document, table_location, item_type="table", folder="tables")

    pdf.close()

## Step 10 - Extract Word Content
Word files are processed for paragraphs and tables.
Both are converted into text chunks so they can be searched through embeddings.

In [14]:
def extract_docx(document, items):
    word_doc = Document(document["filepath"])

    paragraph_text = "\n".join(paragraph.text for paragraph in word_doc.paragraphs)
    add_chunks(items, paragraph_text, document, "Document text")

    for table_number, table in enumerate(word_doc.tables, start=1):
        rows = [[cell.text for cell in row.cells] for row in table.rows]
        add_chunks(items, table_to_text(rows), document, f"Table {table_number}", item_type="table")

## Step 11 - Extract PowerPoint Content
PowerPoint files are processed slide by slide.
Slide text and slide tables are saved as separate searchable chunks.

In [15]:
def extract_pptx(document, items):
    presentation = Presentation(document["filepath"])

    for slide_number, slide in enumerate(presentation.slides, start=1):
        slide_text = []
        slide_tables = []

        for shape in slide.shapes:
            if getattr(shape, "text", "").strip():
                slide_text.append(shape.text)

            if getattr(shape, "has_table", False):
                rows = [[cell.text for cell in row.cells] for row in shape.table.rows]
                slide_tables.append(rows)

        add_chunks(items, "\n".join(slide_text), document, f"Slide {slide_number}")

        for table_number, rows in enumerate(slide_tables, start=1):
            location = f"Slide {slide_number}, Table {table_number}"
            add_chunks(items, table_to_text(rows), document, location, item_type="table")

## Step 12 - Extract Excel And CSV Content
Excel files are processed one sheet at a time, while CSV files are treated as one table.
Each table is converted into text so it can use the same embedding path as other text.

In [16]:
def extract_excel(document, items):
    sheets = pd.read_excel(document["filepath"], sheet_name=None)

    for sheet_name, dataframe in sheets.items():
        add_chunks(items, dataframe_to_text(dataframe), document, f"Sheet {sheet_name}", item_type="table")


def extract_csv(document, items):
    dataframe = pd.read_csv(document["filepath"])
    add_chunks(items, dataframe_to_text(dataframe), document, "CSV table", item_type="table")

## Step 13 - Run Extraction
This cell chooses the right extractor for each uploaded file type.
All extracted records are combined into one `items` list for embedding.

In [17]:
extractors = {
    ".pdf": extract_pdf,
    ".docx": extract_docx,
    ".pptx": extract_pptx,
    ".xlsx": extract_excel,
    ".xls": extract_excel,
    ".csv": extract_csv,
}

items = []

for document in documents:
    print("Processing", document["file_name"])
    extractors[document["extension"]](document, items)

print(f"Extracted {len(items)} item(s)")
print(json.dumps(items[:2], indent=2))

Processing cc1.pdf


cc1.pdf: 100%|██████████| 25/25 [00:42<00:00,  1.71s/it]

Processing 5 PPT Format.pptx
Extracted 249 item(s)
[
  {
    "filepath": "data/uploads/cc1.pdf",
    "file_name": "cc1.pdf",
    "extension": ".pdf",
    "document_id": "ae0bfbc23b967b5fbfdf90ae505d2ae1f96e3c4e5f518d59869e71b9ae8a3f8e",
    "page": "Page 1",
    "location": "Page 1",
    "type": "text",
    "text": "Cloud Computing \u00a9 Vijaya academy                                    1                       \n  \n \nSubject \u2013 CLOUD COMPUTING \n4th Year ENTC  \nUnit \u2013 1 Part 1 \n \n \n \nJoin Now  \n \n \n \n \nJoin Now  \n \n \n \nJoin Now  \n \n \n \nDownload Now \n \n \n \n \n \n \n \nVIJAYA ACADEMY",
    "path": "data/text/cc1.pdf_text_Page_1_0.txt"
  },
  {
    "filepath": "data/uploads/cc1.pdf",
    "file_name": "cc1.pdf",
    "extension": ".pdf",
    "document_id": "ae0bfbc23b967b5fbfdf90ae505d2ae1f96e3c4e5f518d59869e71b9ae8a3f8e",
    "page": "Page 1",
    "location": "Page 1",
    "type": "ocr",
    "text": "@ VIJAYA ACADEMY\nBeyond the Classroom\n\nVIJAYA ACADEMY

## Step 14 - Load CLIP
CLIP is used to create embeddings for both text and images.
The model runs on GPU if available, otherwise it falls back to CPU.

In [18]:
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"
device = "cuda" if torch.cuda.is_available() else "cpu"

clip_model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(device).eval()
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
embedding_vector_dimension = 512

print(f"CLIP loaded on {device}")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


CLIP loaded on cpu


## Step 15 - Define Embedding Functions
Text/table/OCR chunks use CLIP text embeddings.
Extracted images use CLIP image embeddings, so both modalities share the same vector space.

In [19]:
@torch.no_grad()
def embed_text(text):
    inputs = clip_processor(
        text=[text],
        return_tensors="pt",
        padding=True,
        truncation=True,
    )
    inputs = {key: value.to(device) for key, value in inputs.items()}
    vector = clip_model.get_text_features(**inputs).cpu().numpy()[0]
    return (vector / (np.linalg.norm(vector) + 1e-10)).astype(np.float32)


@torch.no_grad()
def embed_image(path):
    image = Image.open(path).convert("RGB")
    inputs = clip_processor(images=image, return_tensors="pt")
    inputs = {key: value.to(device) for key, value in inputs.items()}
    vector = clip_model.get_image_features(**inputs).cpu().numpy()[0]
    return (vector / (np.linalg.norm(vector) + 1e-10)).astype(np.float32)

## Step 16 - Embed Extracted Items
This cell adds an embedding vector to each extracted item.
Items that fail embedding are skipped so one bad file does not stop the whole pipeline.

In [20]:
for item in tqdm(items, desc="Embedding"):
    try:
        if item["type"] == "image":
            item["embedding"] = embed_image(item["path"])
        else:
            item["embedding"] = embed_text(item.get("text", ""))
    except Exception as error:
        print(f"Skipped {item.get('file_name')} {item.get('location')}: {error}")

items = [item for item in items if "embedding" in item]

print(f"Embedded {len(items)} item(s)")

Embedding: 100%|██████████| 249/249 [00:29<00:00,  8.50it/s]

Embedded 249 item(s)


## Step 17 - Connect To Pinecone
This cell connects to Pinecone using `PINECONE_API_KEY` from `.env`.
It also reads optional index, namespace, cloud, and region settings.

In [21]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

if not PINECONE_API_KEY:
    raise ValueError("Set PINECONE_API_KEY in .env")

INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "multimodal-rag")
NAMESPACE = os.getenv("PINECONE_NAMESPACE", "default")
PINECONE_CLOUD = os.getenv("PINECONE_CLOUD", "aws")
PINECONE_REGION = os.getenv("PINECONE_REGION", "us-east-1")

pc = Pinecone(api_key=PINECONE_API_KEY)

print("Connected to Pinecone")

Connected to Pinecone


## Step 18 - Create Pinecone Index
Create the Pinecone index if it does not already exist.
The index dimension is 512 because CLIP produces 512-dimensional vectors.

In [22]:
def pinecone_index_names():
    return [index["name"] if isinstance(index, dict) else index.name for index in pc.list_indexes()]


def pinecone_index_ready(name):
    status = pc.describe_index(name).status
    return status.get("ready", False) if isinstance(status, dict) else getattr(status, "ready", False)


if INDEX_NAME not in pinecone_index_names():
    pc.create_index(
        name=INDEX_NAME,
        vector_type="dense",
        dimension=embedding_vector_dimension,
        metric="cosine",
        spec=ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION),
        deletion_protection="disabled",
    )

    while not pinecone_index_ready(INDEX_NAME):
        time.sleep(5)

index = pc.Index(INDEX_NAME)

print(f"Pinecone index ready: {INDEX_NAME}")

Pinecone index ready: multimodal-rag


## Step 19 - Upload Vectors
This cell clears the selected namespace and uploads embedded items in batches.
Each vector includes metadata for file name, location, type, text, and source path.

In [23]:
try:
    index.delete(delete_all=True, namespace=NAMESPACE)
except Exception:
    pass

for start in range(0, len(items), 100):
    batch = []

    for vector_id, item in enumerate(items[start:start + 100], start):
        metadata = {
            key: item.get(key)
            for key in ["document_id", "file_name", "page", "location", "type", "path", "text"]
            if item.get(key) is not None
        }

        batch.append({
            "id": str(vector_id),
            "values": item["embedding"].tolist(),
            "metadata": metadata,
        })

    index.upsert(vectors=batch, namespace=NAMESPACE)
    print(f"Uploaded {min(start + 100, len(items))}/{len(items)}")

Uploaded 100/249
Uploaded 200/249
Uploaded 249/249


## Step 20 - Enter Query
Set the question you want to ask about the uploaded documents.
Only this cell usually needs to change when you ask a new question.

In [24]:
query = "What is the definition of cloud computing as per NIST?"

print("Query:", query)

Query: What is the definition of cloud computing as per NIST?


## Step 21 - Retrieve From Pinecone
The query is embedded with CLIP and searched against Pinecone.
The top matches are returned with metadata so they can become context and citations.

In [25]:
query_embedding = embed_text(query)

retrieved = index.query(
    vector=query_embedding.tolist(),
    top_k=15,
    namespace=NAMESPACE,
    include_metadata=True,
).matches

print(f"Retrieved {len(retrieved)} candidate(s)")

Retrieved 15 candidate(s)


## Step 22 - Define Re-ranker Helpers
These helpers normalize Pinecone match objects and scores.
They make the re-ranking code work whether Pinecone returns objects or dictionaries.

In [26]:
def get_metadata(match):
    if hasattr(match, "metadata"):
        return match.metadata or {}

    return match.get("metadata", {}) if isinstance(match, dict) else {}


def get_score(match):
    if hasattr(match, "score") and match.score is not None:
        return float(match.score)

    return float(match.get("score", 0.0)) if isinstance(match, dict) else 0.0

## Step 23 - Re-rank Results
The CrossEncoder re-ranker improves the order of retrieved text, table, and OCR matches.
Image-only matches keep their Pinecone similarity score.

In [27]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", max_length=512)


def rerank(query, matches, top_k=5):
    rows = []
    text_pairs = []
    text_row_ids = []

    for row_id, match in enumerate(matches):
        metadata = get_metadata(match)
        text = metadata.get("text", "")
        item_type = metadata.get("type")

        rows.append({
            "match": match,
            "metadata": metadata,
            "pinecone_score": get_score(match),
            "rerank_score": None,
        })

        if item_type in ["text", "table", "ocr"] and text.strip():
            text_pairs.append((query, text))
            text_row_ids.append(row_id)

    if text_pairs:
        scores = reranker.predict(text_pairs, show_progress_bar=False)

        for row_id, score in zip(text_row_ids, scores):
            rows[row_id]["rerank_score"] = float(score)

    for row in rows:
        row["final_score"] = row["rerank_score"] if row["rerank_score"] is not None else row["pinecone_score"]

    return sorted(rows, key=lambda row: row["final_score"], reverse=True)[:top_k]


reranked_results = rerank(query, retrieved)

for rank, result in enumerate(reranked_results, start=1):
    metadata = result["metadata"]
    print(rank, metadata.get("file_name"), metadata.get("location"), metadata.get("type"), round(result["final_score"], 4))

1 cc1.pdf Page 4 ocr 7.1181
2 cc1.pdf Page 11, Table 1 table 3.193
3 cc1.pdf Page 12 text 3.0167
4 cc1.pdf Page 2, Table 1 table 1.9849
5 cc1.pdf Page 2 text 0.7615


## Step 24 - Build Context And Citations
This cell turns re-ranked matches into LLM context and source lines.
Text is sent as evidence, while image paths are collected for optional vision-model use.

In [28]:
def build_context(results):
    context_parts = []
    image_paths = []
    source_lines = []

    for source_number, result in enumerate(results, start=1):
        metadata = result["metadata"]
        file_name = metadata.get("file_name", "Unknown")
        location = metadata.get("location", metadata.get("page", "Unknown"))
        item_type = metadata.get("type", "document")
        text = metadata.get("text", "")

        source_lines.append(f"[{source_number}] {file_name} - {location} - {item_type}")

        if text.strip():
            context_parts.append(
                f"[Retrieved Source {source_number}]\n"
                f"File: {file_name}\n"
                f"Location: {location}\n"
                f"Type: {item_type}\n\n"
                f"{text}"
            )

        if item_type == "image" and metadata.get("path") and Path(metadata["path"]).exists():
            image_paths.append(metadata["path"])
            context_parts.append(
                f"[Retrieved Image {source_number}]\n"
                f"File: {file_name}\n"
                f"Location: {location}\n"
                f"Type: image"
            )

    return "\n\n".join(context_parts), image_paths, "\n".join(source_lines)


context, image_paths, sources = build_context(reranked_results)

print(f"Context characters: {len(context)}")
print(f"Images available: {len(image_paths)}")

Context characters: 2036
Images available: 0


## Step 25 - Generate Final Answer
Ollama receives the question and retrieved context, then generates the answer.
Sources are shown in the next cell so the citation list stays separate and ordered.


In [32]:
MAX_CONTEXT_CHARS = 8000
MAX_IMAGES_FOR_ANSWER = 0
OLLAMA_MODEL = "qwen2.5:3b"

prompt = f"""
You are a document question-answering assistant.
Answer using only the retrieved context and images.
If the answer is not present, say it was not found.
Create a Sources section.

User question:
{query}

Retrieved context:
{context[:MAX_CONTEXT_CHARS]}
"""

message = {
    "role": "user",
    "content": prompt,
}

if MAX_IMAGES_FOR_ANSWER:
    message["images"] = image_paths[:MAX_IMAGES_FOR_ANSWER]

print("Generating answer...")

answer = ollama.Client(timeout=120).chat(
    model=OLLAMA_MODEL,
    messages=[message],
    options={"num_predict": 512, "temperature": 0.1},
    keep_alive="5m",
)["message"]["content"]

print("Answer")
print("=" * 60)
print(answer)


Generating answer...
Answer
According to the National Institute of Standards and Technology (NIST), cloud computing is a model for enabling convenient, on-demand network access to a shared pool of configurable computing resources that can be rapidly provisioned and released with minimal management effort or service provider interaction.

The main characteristics of NIST's cloud computing model include:

1. **On-Demand Self-Service**: Users can request services such as application software, storage space, and processing power without human intervention.
2. **Broad Network Access**: Services are accessed over a network (typically the Internet) from any location with an internet connection.
3. **Resource Pooling**: The computing resources are pooled together to serve multiple users or applications. Resources can be rapidly provisioned and released with minimal management effort.
4. **Measured Service**: Resource usage is metered, providing cost control and transparency.

Cloud computing l